In [ ]:
# Notebook version of plot_type_distributions.py
# Bar charts: estimated EV charger penetration per dwelling/consumption type
# (DEF_KODE) and per heating type (VARME), RF vs XGBoost, one figure each.
#
# Expects rf/xgb_EV_hus_type.csv and rf/xgb_EV_varme_type.csv in MSc-Thesis/data.

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# anchor paths: __file__ doesn't exist in a notebook kernel, so fall back to
# the kernel's working directory (VS Code starts it in this notebook's folder, src)
ROOT = (Path(__file__).resolve().parent.parent if "__file__" in globals()
        else Path.cwd().parent)                 # MSc-Thesis/
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

# thesis-readable font sizes (figures get scaled down when placed in Overleaf)
plt.rcParams.update({
    "font.size": 13,          # base size (fallback for anything not set below)
    "axes.titlesize": 15,     # panel title
    "axes.labelsize": 13,     # x/y axis labels
    "xtick.labelsize": 12,    # x tick numbers
    "ytick.labelsize": 12,    # category names on the y axis
    "legend.fontsize": 12,
})

DEF_LABELS = {
    100: "Dwellings (unspecified)",
    111: "Apartment, no electric heating",
    112: "Apartment, electric heating",
    113: "Apartment, heat pump",
    121: "Single-family house, no electric heating",
    122: "Single-family house, electric heating",
    123: "Single-family house, heat pump",
    131: "Summer house, no electric heating",
    132: "Summer house, electric heating",
    133: "Summer house, heat pump",
    134: "Allotment garden",
}
VARME_LABELS = {
    0: "Not stated",
    1: "None",
    2: "Other heating (district/gas/oil)",
    3: "Electric heating",
    4: "Heat pump",
    5: "Mixed",
}

MIN_METERS = 2000   # categories below this are dropped (footnoted in the caption)


def load(prefix, code_col, labels, threshold):
    col = f"n_evs_p{threshold}"
    out = {}
    for m in ("rf", "xgb"):
        df = pd.read_csv(DATA_DIR / f"{m}_EV_{prefix}.csv")
        df["label"] = df[code_col].map(labels)
        df["pen"] = df[col] / df["n_meters"] * 100
        out[m] = df.set_index("label")[["pen", "n_meters"]]
    merged = out["rf"].join(out["xgb"], lsuffix="_rf", rsuffix="_xgb")
    merged = merged[merged["n_meters_rf"] >= MIN_METERS]
    return merged.sort_values("pen_rf")


def main(threshold):
    hus = load("hus_type", "DEF_KODE", DEF_LABELS, threshold)
    varme = load("varme_type", "VARME", VARME_LABELS, threshold)

    # one figure per category type, saved separately
    panels = [
        (hus, "Dwelling / consumption type (DEF_KODE)", "dwelling"),
        (varme, "Heating type (VARME)", "heating"),
    ]
    for data, title, suffix in panels:
        fig, ax = plt.subplots(figsize=(8, 6))
        y = np.arange(len(data))
        ax.barh(y + 0.2, data["pen_rf"], height=0.4, label="Random Forest")
        ax.barh(y - 0.2, data["pen_xgb"], height=0.4, label="XGBoost")
        ax.set_yticks(y, data.index)
        ax.set_xlabel(f"EV charger penetration (% of meters), "
                      f"$\\tau={threshold/100:.2f}$")
        ax.set_title(title)
        # annotate meter counts for context
        for yi, (_, row) in zip(y, data.iterrows()):
            ax.annotate(f"n={int(row['n_meters_rf']):,}",
                        xy=(max(row['pen_rf'], row['pen_xgb']), yi),
                        xytext=(4, 0), textcoords="offset points",
                        va="center", fontsize=12, color="grey")
        # headroom past the longest bar so the n= annotations stay inside the axes
        ax.set_xlim(0, data[["pen_rf", "pen_xgb"]].max().max() * 1.1)
        ax.legend(loc="lower right")
        plt.tight_layout()
        out = RESULTS_DIR / f"type_distributions_{suffix}_p{threshold}.pdf"
        plt.savefig(out, bbox_inches="tight", dpi=300)
        plt.show()
        plt.close(fig)
        print(f"saved {out}")


main(90)
